# Séance 10 – La gestion des exceptions
### Lab : capturer, lever et créer des exceptions

| | |
|---|---|
| **Durée estimée** | 60 min (2 × 30 min) + bonus |
| **Projet fil rouge** | GestParc – étape 7 : exceptions métier |

**Objectifs**
- Capturer des exceptions précises avec try / except / else / finally
- Exploiter la hiérarchie des exceptions
- Utiliser assert et raise à bon escient
- Créer des exceptions métier et chaîner les erreurs

**Mode d'emploi**
- Exécuter une cellule : **Maj + Entrée** ; en cas de comportement étrange : *Redémarrer le noyau et tout exécuter*.
- Complétez les cellules marquées `# TODO`, puis exécutez la cellule **✅ Vérification** qui suit : aucune erreur = réponse correcte.
- Difficulté : ⭐ facile · ⭐⭐ intermédiaire · ⭐⭐⭐ avancé · Indices dans les blocs 💡 repliables.

---
## Partie 1 – Capturer des exceptions

> **Rappel**
> ```python
> try:
>     operation_risquee()
> except (KeyError, ValueError) as exc:   # une ou plusieurs exceptions précises
>     traiter(exc)
> else:
>     # seulement si aucune exception
> finally:
>     # toujours exécuté
> ```
> - Hiérarchie : `LookupError` ⊃ `KeyError`, `IndexError` · `ArithmeticError` ⊃ `ZeroDivisionError` · `OSError` ⊃ `FileNotFoundError`

### Exemple guidé : moyenne de mesures robuste

In [ ]:
def average_load(raw_values: list[str]) -> float:
    """Moyenne des valeurs numériques ; les valeurs illisibles sont ignorées."""
    values = []
    for raw in raw_values:
        try:
            values.append(float(raw))
        except ValueError:
            print(f"  valeur ignorée : {raw!r}")
    try:
        return sum(values) / len(values)
    except ZeroDivisionError:
        return 0.0


print(average_load(["0.42", "0.91", "erreur", "0.65"]))
print(average_load(["N/A"]))

### Exercice 1 ⭐ – Conversion sûre d'une saisie

Écrivez `safe_int(text: str, default: int | None = None) -> int | None` qui convertit `text` en entier (espaces tolérés)
et renvoie `default` si la conversion échoue. Puis `safe_ratio(a: float, b: float) -> float | None` qui renvoie `a / b`, ou `None` si `b` vaut 0.

In [ ]:
def safe_int(text: str, default: int | None = None) -> int | None:
    """Convertit un texte en entier, ou renvoie default."""
    # TODO


def safe_ratio(a: float, b: float) -> float | None:
    """Renvoie a / b, ou None si b est nul."""
    # TODO


print(safe_int(" 42 "), safe_int("4,2"), safe_int("abc", 0), safe_ratio(3, 4), safe_ratio(1, 0))

<details><summary>💡 Indice</summary>

Ne capturez que `ValueError` dans `safe_int` et `ZeroDivisionError` dans `safe_ratio`.

</details>

In [ ]:
# ✅ Vérification
assert safe_int(" 42 ") == 42 and safe_int("4,2") is None and safe_int("abc", 0) == 0
assert safe_ratio(3, 4) == 0.75 and safe_ratio(1, 0) is None
print("✅ Exercice 1 réussi")

### Exercice 2 ⭐⭐ – Charger une configuration en collectant les erreurs

Écrivez `load_settings(raw: dict[str, str], schema: dict[str, type]) -> tuple[dict, list[str]]` :
pour chaque clé du `schema` (clé → type attendu : `int`, `float` ou `str`), lisez `raw[clé]` et convertissez-la.
- clé absente → erreur `"<clé> : absent"` ;
- conversion impossible → erreur `"<clé> : valeur invalide '<valeur>'"`.

La fonction renvoie le dictionnaire des valeurs valides et la liste des erreurs (dans l'ordre du schéma).

In [ ]:
def load_settings(raw: dict[str, str], schema: dict[str, type]) -> tuple[dict, list[str]]:
    """Convertit une configuration texte selon un schéma ; renvoie (valeurs, erreurs)."""
    # TODO


raw = {"host": "srv-db-01", "port": "5432", "timeout": "2,5", "pool_size": "10"}
schema = {"host": str, "port": int, "timeout": float, "pool_size": int, "ssl_mode": str}
settings, errors = load_settings(raw, schema)
print(settings, errors)

<details><summary>💡 Indice</summary>

Les types sont des fonctions de conversion : `expected(raw[key])` vaut `int("5432")` quand `expected` est `int`. Deux `except` distincts.

</details>

In [ ]:
# ✅ Vérification
assert settings == {"host": "srv-db-01", "port": 5432, "pool_size": 10}
assert errors == ["timeout : valeur invalide '2,5'", "ssl_mode : absent"]
assert load_settings({}, {}) == ({}, [])
print("✅ Exercice 2 réussi")

### Exercice 3 ⭐⭐ – else et finally autour d'une connexion

La classe `FakeConnection` simule une connexion à une base de données. Écrivez `run_query(conn, query) -> str | None` qui :
1. ouvre la connexion (`conn.open()`) ;
2. exécute la requête (`conn.execute(query)`, qui peut lever `TimeoutError` ou `PermissionError`) ;
3. en cas de `TimeoutError`, ajoute `"timeout"` au journal et renvoie `None` ;
4. **si tout s'est bien passé**, ajoute `"ok"` au journal (`else`) et renvoie le résultat ;
5. **dans tous les cas**, ferme la connexion (`finally`). `PermissionError` n'est pas capturée : elle remonte à l'appelant.

In [ ]:
class FakeConnection:
    """Connexion simulée (fournie)."""

    def __init__(self) -> None:
        self.log: list[str] = []
        self.is_open = False

    def open(self) -> None:
        self.is_open = True
        self.log.append("open")

    def execute(self, query: str) -> str:
        if "slow" in query:
            raise TimeoutError("requête trop longue")
        if query.startswith("DROP"):
            raise PermissionError("droits insuffisants")
        return f"résultat de {query}"

    def close(self) -> None:
        self.is_open = False
        self.log.append("close")


def run_query(conn: FakeConnection, query: str) -> str | None:
    """Exécute une requête en garantissant la fermeture de la connexion."""
    # TODO


c = FakeConnection()
print(run_query(c, "SELECT 1"), run_query(c, "SELECT slow"), c.log)

<details><summary>💡 Indice</summary>

Le `finally` s'exécute même après un `return` dans `except` ou `else`, et même si l'exception n'est pas capturée.

</details>

In [ ]:
# ✅ Vérification
c = FakeConnection()
assert run_query(c, "SELECT 1") == "résultat de SELECT 1" and c.log == ["open", "ok", "close"]
c = FakeConnection()
assert run_query(c, "SELECT slow") is None and c.log == ["open", "timeout", "close"]
c = FakeConnection()
try:
    run_query(c, "DROP TABLE users")
    raise AssertionError("PermissionError aurait dû remonter")
except PermissionError:
    pass
assert c.log == ["open", "close"] and not c.is_open
print("✅ Exercice 3 réussi")

### Exercice 4 ⭐⭐ – Capturer par famille d'exceptions

La fonction `lookup(data, path)` suit un chemin de clés et d'indices dans une structure imbriquée (fournie).
Écrivez `safe_lookup(data, path, default=None)` qui renvoie `default` en cas de clé **ou** d'indice introuvable,
en capturant **une seule** famille d'exceptions. Les autres erreurs (par exemple `TypeError`) doivent remonter.

Puis classez dans `families` le nom de la **classe mère directe** (`type(exc).__mro__[1].__name__`) de chaque exception levée par les opérations de `operations`.

In [ ]:
def lookup(data, path: list):
    """Suit un chemin de clés / indices (fourni)."""
    for step in path:
        data = data[step]
    return data


def safe_lookup(data, path: list, default=None):
    """Comme lookup, mais renvoie default si une clé ou un indice est absent."""
    # TODO


infra = {"sites": [{"name": "Tunis", "servers": ["srv-web-01", "srv-db-01"]}, {"name": "Sfax", "servers": []}]}
print(safe_lookup(infra, ["sites", 0, "servers", 1]), safe_lookup(infra, ["sites", 1, "servers", 0], "aucun"))

operations = [lambda: 1 / 0, lambda: [1][5], lambda: {}["x"], lambda: int("x"), lambda: open("/introuvable.txt")]
families = []
# TODO
print(families)

<details><summary>💡 Indice</summary>

`except LookupError` capture `KeyError` et `IndexError`. Indicer une liste par une chaîne lève `TypeError`.

</details>

In [ ]:
# ✅ Vérification
assert safe_lookup(infra, ["sites", 0, "servers", 1]) == "srv-db-01"
assert safe_lookup(infra, ["sites", 9]) is None and safe_lookup(infra, ["region"], "?") == "?"
try:
    safe_lookup(infra, ["sites", "zero"])
    raise AssertionError("TypeError aurait dû remonter")
except TypeError:
    pass
assert families == ["ArithmeticError", "LookupError", "LookupError", "Exception", "OSError"]
print("✅ Exercice 4 réussi")

---
## Partie 2 – Assertions, raise et exceptions personnalisées

> **Rappel**
> - `assert condition, "message"` : invariants internes (désactivables avec `python -O`)
> - `raise ValueError("message précis")` · `raise` seul dans un `except` relance l'exception courante
> - `class MonErreur(Exception): ...` · une base métier commune · attributs supplémentaires via `__init__` + `super().__init__(message)`
> - `raise NouvelleErreur(...) from exc` conserve la cause (`__cause__`)

### Exemple guidé : un quota de stockage

In [ ]:
class QuotaExceededError(Exception):
    """Le quota de stockage d'un utilisateur est dépassé."""

    def __init__(self, user: str, requested_gb: float, available_gb: float) -> None:
        super().__init__(f"{user} : {requested_gb} Go demandés, {available_gb} Go disponibles")
        self.user = user
        self.missing_gb = requested_gb - available_gb


def upload(user: str, size_gb: float, used_gb: float, quota_gb: float = 50) -> float:
    if size_gb <= 0:
        raise ValueError(f"taille invalide : {size_gb}")
    available = quota_gb - used_gb
    if size_gb > available:
        raise QuotaExceededError(user, size_gb, available)
    return used_gb + size_gb


for size in [5, 20, -1]:
    try:
        print("Nouvel espace utilisé :", upload("amira", size, used_gb=40))
    except QuotaExceededError as exc:
        print("Refus :", exc, f"(manque {exc.missing_gb} Go)")
    except ValueError as exc:
        print("Erreur :", exc)

### Exercice 5 ⭐ – Assertions de préconditions

Ajoutez des **assertions** à `split_bandwidth(total_mbps, weights)` qui répartit une bande passante proportionnellement à des poids :
- `total_mbps` est strictement positif ;
- `weights` n'est pas vide et tous les poids sont positifs ou nuls ;
- la somme des poids est strictement positive.
Chaque assertion a un message explicite.

In [ ]:
def split_bandwidth(total_mbps: float, weights: dict[str, float]) -> dict[str, float]:
    """Répartit une bande passante proportionnellement aux poids."""
    # TODO : assertions
    total_weight = sum(weights.values())
    return {name: round(total_mbps * w / total_weight, 1) for name, w in weights.items()}


print(split_bandwidth(1000, {"voip": 3, "web": 5, "backup": 2}))

<details><summary>💡 Indice</summary>

`all([...])` vaut `True` si toutes les conditions de la liste sont vraies.

</details>

In [ ]:
# ✅ Vérification
assert split_bandwidth(1000, {"voip": 3, "web": 5, "backup": 2}) == {"voip": 300.0, "web": 500.0, "backup": 200.0}
for args in [(0, {"a": 1}), (100, {}), (100, {"a": -1, "b": 2}), (100, {"a": 0})]:
    try:
        split_bandwidth(*args)
        raise RuntimeError(f"assertion attendue pour {args}")
    except AssertionError as exc:
        assert str(exc), "chaque assertion doit avoir un message"
print("✅ Exercice 5 réussi")

### Exercice 6 ⭐⭐ – raise dans les propriétés

Reprenez l'onduleur de S7 : au lieu de **ramener** la charge dans [0, 100], le setter doit maintenant **lever** :
- `TypeError` si la valeur n'est ni un `int` ni un `float` (attention : `bool` est une sous-classe d'`int` et doit être refusé) ;
- `ValueError` si la valeur est hors de [0, 100].

`discharge(minutes)` lève `ValueError` si `minutes` est négatif, et ne descend pas sous 0 % (pas d'erreur dans ce cas).

In [ ]:
class UPS:
    """Onduleur de salle serveur."""

    def __init__(self, name: str, charge: float = 100.0) -> None:
        self.name = name
        self.charge = charge

    @property
    def charge(self) -> float:
        return self._charge

    @charge.setter
    def charge(self, value: float) -> None:
        # TODO : TypeError, ValueError
        self._charge = float(value)

    def discharge(self, minutes: int, rate: float = 1.5) -> None:
        # TODO
        ...


ups = UPS("ups-a")
ups.discharge(80)
print(ups.charge)

<details><summary>💡 Indice</summary>

Testez le type **avant** de comparer la valeur : `"50" <= 100` lèverait lui-même une `TypeError` peu explicite.

</details>

In [ ]:
# ✅ Vérification
u = UPS("t", 50)
u.discharge(10)
assert u.charge == 35.0
u.discharge(100)
assert u.charge == 0.0
for bad, expected in [(120, ValueError), (-1, ValueError), ("50", TypeError), (True, TypeError), (None, TypeError)]:
    try:
        UPS("x", bad)
        raise AssertionError(f"{expected.__name__} attendue pour {bad!r}")
    except expected:
        pass
try:
    u.discharge(-5)
    raise AssertionError("ValueError attendue")
except ValueError:
    pass
print("✅ Exercice 6 réussi")

### Exercice 7 ⭐⭐ – Exceptions d'un pipeline de déploiement

Créez la hiérarchie :
- `DeploymentError(Exception)` : base ;
- `BuildError(DeploymentError)` ;
- `TestFailureError(DeploymentError)` avec un attribut `failed` (nombre de tests en échec) et le message `"3 test(s) en échec"` ;
- `HealthCheckError(DeploymentError)`.

Complétez `deploy(version, build_ok, failed_tests, healthy)` qui lève l'exception adaptée à la **première** étape en échec,
puis `release(...)` qui appelle `deploy` et renvoie : `"v1.4.0 déployée"`, `"v1.4.0 bloquée : <message>"` pour toute `DeploymentError`,
et ajoute `" (rollback)"` uniquement pour une `HealthCheckError`.

In [ ]:
# TODO : DeploymentError, BuildError, TestFailureError, HealthCheckError


def deploy(version: str, build_ok: bool, failed_tests: int, healthy: bool) -> None:
    """Simule les étapes d'un déploiement."""
    # TODO


def release(version: str, build_ok: bool = True, failed_tests: int = 0, healthy: bool = True) -> str:
    """Déploie et renvoie un compte rendu."""
    # TODO


print(release("v1.4.0"), release("v1.4.1", failed_tests=3), release("v1.4.2", healthy=False), sep="\n")

<details><summary>💡 Indice</summary>

Ordre des `except` : la classe la plus précise d'abord (`HealthCheckError`), la classe de base ensuite.

</details>

In [ ]:
# ✅ Vérification
assert release("v1.4.0") == "v1.4.0 déployée"
assert release("v1.4.1", failed_tests=3) == "v1.4.1 bloquée : 3 test(s) en échec"
assert release("v1.4.2", healthy=False) == "v1.4.2 bloquée : le contrôle /health ne répond pas (rollback)"
assert release("v1.4.3", build_ok=False, failed_tests=2).startswith("v1.4.3 bloquée : échec de compilation")
try:
    deploy("v", True, 2, True)
except TestFailureError as exc:
    assert exc.failed == 2 and isinstance(exc, DeploymentError)
print("✅ Exercice 7 réussi")

### Exercice 8 ⭐⭐⭐ – Nouvelles tentatives et raise ... from

Écrivez `retry(operation, attempts=3, retry_on=(ConnectionError,))` :
- appelle `operation()` jusqu'à `attempts` fois ;
- renvoie son résultat dès qu'un appel réussit ;
- recommence seulement pour les exceptions de `retry_on` ; toute autre exception remonte **immédiatement** ;
- si toutes les tentatives échouent, lève `RetryError(f"échec après {attempts} tentative(s)")` **chaînée** à la dernière exception (`from`).

`FlakyService` (fourni) échoue un certain nombre de fois avant de répondre.

In [ ]:
class RetryError(Exception):
    """Toutes les tentatives ont échoué."""


class FlakyService:
    """Service instable (fourni) : échoue `failures` fois puis répond."""

    def __init__(self, failures: int, error: type[Exception] = ConnectionError) -> None:
        self.failures = failures
        self.error = error
        self.calls = 0

    def __call__(self) -> str:
        self.calls += 1
        if self.calls <= self.failures:
            raise self.error(f"échec n° {self.calls}")
        return "200 OK"


def retry(operation, attempts: int = 3, retry_on: tuple = (ConnectionError,)):
    """Exécute operation avec plusieurs tentatives."""
    # TODO


print(retry(FlakyService(2)))

<details><summary>💡 Indice</summary>

`except retry_on as exc:` fonctionne avec un tuple d'exceptions. Gardez la dernière exception dans une variable pour le `from`.

</details>

In [ ]:
# ✅ Vérification
s = FlakyService(2)
assert retry(s) == "200 OK" and s.calls == 3
s = FlakyService(5)
try:
    retry(s, attempts=4)
    raise AssertionError("RetryError attendue")
except RetryError as exc:
    assert str(exc) == "échec après 4 tentative(s)" and s.calls == 4
    assert isinstance(exc.__cause__, ConnectionError) and str(exc.__cause__) == "échec n° 4"
s = FlakyService(1, error=PermissionError)
try:
    retry(s)
    raise AssertionError("PermissionError attendue")
except PermissionError:
    assert s.calls == 1
assert retry(FlakyService(1, TimeoutError), retry_on=(TimeoutError, ConnectionError)) == "200 OK"
print("✅ Exercice 8 réussi")

---
### Exercice de synthèse ⭐⭐⭐ – Import robuste d'une liste d'employés

Un export RH contient des lignes `matricule;prénom;nom;service;date_embauche(AAAA-MM-JJ)`.
1. Créez `InvalidRecordError(ValueError)` avec les attributs `line_number` et `reason`, et le message `"ligne 3 : <raison>"`.
2. Écrivez `parse_record(line_number, line) -> dict` qui lève `InvalidRecordError` si :
   - le nombre de champs n'est pas 5 → raison `"5 champs attendus, 4 reçus"` ;
   - le matricule n'est pas `E` suivi de 4 chiffres → `"matricule invalide 'X12'"` ;
   - un champ est vide → `"champ vide"` ;
   - l'année d'embauche n'est pas un entier entre 1990 et 2026 → `"date invalide '...'"` (utilisez `raise ... from` pour une erreur de conversion).
3. Écrivez `import_employees(lines) -> tuple[list[dict], list[str]]` qui ignore les lignes vides, collecte les employés valides et les messages d'erreur, sans jamais s'arrêter.

In [ ]:
class InvalidRecordError(ValueError):
    """Ligne d'import invalide."""
    # TODO


def parse_record(line_number: int, line: str) -> dict:
    """Analyse une ligne de l'export RH."""
    # TODO


def import_employees(lines: list[str]) -> tuple[list[dict], list[str]]:
    """Importe les lignes valides et collecte les erreurs."""
    # TODO


export = [
    "E0001;Pierre;Martin;DSI;2019-03-01",
    "E0002;Sophie;Dubois;Finance",
    "X12;Jean;Moreau;DSI;2021-09-15",
    "",
    "E0004;Claire;;RH;2020-01-06",
    "E0005;Thomas;Laurent;DSI;deux-mille",
    "E0006;Nicolas;Petit;Achats;1985-05-20",
    "E0007;Julie;Robert;DSI;2024-02-01",
]
employees, errors = import_employees(export)
print([e["id"] for e in employees])
print("\n".join(errors))

<details><summary>💡 Indice</summary>

Vérifiez les champs vides avant le matricule (l'ordre des contrôles détermine le message). `all(fields)` est faux si un champ est vide.

</details>

In [ ]:
# ✅ Vérification
assert [e["id"] for e in employees] == ["E0001", "E0007"]
assert errors == [
    "ligne 2 : 5 champs attendus, 4 reçus",
    "ligne 3 : matricule invalide 'X12'",
    "ligne 5 : champ vide",
    "ligne 6 : date invalide 'deux-mille'",
    "ligne 7 : date invalide '1985-05-20'",
]
try:
    parse_record(6, "E0005;Thomas;Laurent;DSI;deux-mille")
except InvalidRecordError as exc:
    assert isinstance(exc, ValueError) and exc.line_number == 6 and isinstance(exc.__cause__, ValueError)
print("✅ Exercice de synthèse réussi")